In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
df = pd.read_csv("glass.csv")

In [3]:
print(df.shape)
print(df.columns)
print(df.head())

(214, 10)
Index(['RI', 'Na', 'Mg', 'Al', 'Si', 'K', 'Ca', 'Ba', 'Fe', 'Type'], dtype='object')
        RI     Na    Mg    Al     Si     K    Ca   Ba   Fe  Type
0  1.52101  13.64  4.49  1.10  71.78  0.06  8.75  0.0  0.0     1
1  1.51761  13.89  3.60  1.36  72.73  0.48  7.83  0.0  0.0     1
2  1.51618  13.53  3.55  1.54  72.99  0.39  7.78  0.0  0.0     1
3  1.51766  13.21  3.69  1.29  72.61  0.57  8.22  0.0  0.0     1
4  1.51742  13.27  3.62  1.24  73.08  0.55  8.07  0.0  0.0     1


In [4]:
df["y"] = (df["Type"] == 1).astype(int)
df = df.drop(columns=["Type"])

In [5]:
X = df.drop(columns=["y"]).values
y = df["y"].values

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [7]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [8]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [9]:
def predict_proba(X, w, b):
    z = X @ w + b
    p = sigmoid(z)
    return p

In [10]:
def loss(y, p):
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

In [20]:
def update_weights(X, y, w, b, lr):
    p = predict_proba(X, w, b)
    error = p - y
    w = w - lr * (X.T @ error) / len(y)
    b = b - lr * np.mean(error)
    return w, b


In [21]:
def predict_label(p, threshold=0.5):
    return (p >= threshold).astype(int)

In [24]:

w = np.zeros(X_train.shape[1])
b = 0.0
lr = 0.1
epochs = 100

for i in range(epochs):
    w, b = update_weights(X_train, y_train, w, b, lr)
    if i % 10 == 0:
        p_current = predict_proba(X_train, w, b)
        current_loss = loss(y_train, p_current)
        print(f"Epoch{i}:Loss ={current_loss:.4f}")

Epoch0:Loss =0.6822
Epoch10:Loss =0.6107
Epoch20:Loss =0.5748
Epoch30:Loss =0.5529
Epoch40:Loss =0.5379
Epoch50:Loss =0.5269
Epoch60:Loss =0.5184
Epoch70:Loss =0.5116
Epoch80:Loss =0.5060
Epoch90:Loss =0.5014


In [25]:
probabilities = predict_proba(X_test, w, b)
predictions_50 = predict_label(probabilities, threshold=0.5)
predictions_70 = predict_label(probabilities, threshold=0.7)
print("First 5 probabilities:", probabilities[:5])
print("First 5 predictions (0.5):", predictions_50[:5])

First 5 probabilities: [0.46596617 0.02699184 0.60769832 0.02225283 0.30880036]
First 5 predictions (0.5): [0 0 1 0 0]


In [27]:
print('First 5 predictions(0.7):',predictions_70[:5])

First 5 predictions(0.7): [0 0 0 0 0]


In [ ]:
#higher threshold is better because we need more safety in case of glass quality control
#we cant let less safe samples slide
#so criteria should be stricter and hence higher threshold is more preferable

In [26]:
#this differs from perceptron because perceptron uses step function
#here we need sigmoid because we need probability of something
#simgmoid matters because it helps provide probabilities between 0 and 1 which can then be used to decide
#0 or 1 depending on threshold value set
#problem still remaining is that this will work if dependency is linear
#in case of non linear dependency there will be problem